# 02 — Exploratory Data Analysis

This notebook explores the processed tiles to understand:
1. Class distributions in BGT vs BRT
2. Geometric properties (area, perimeter, vertex count)
3. How many BGT polygons survive into BRT (selection rate)
4. Spatial overlap patterns between BGT and BRT

Run **01_data_prep.ipynb** first.


## 0. Shared config (copy of CONFIG from notebook 01)

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root":   Path("processed"),
    "bgt_class_col": "plus_type",       # must match 01_data_prep.ipynb
    "brt_class_col": "typewegde",
    "crs":           "EPSG:28992",

    # How many sample tiles to load for EDA (keep small for speed)
    "eda_sample_tiles": 20,

    # IoU threshold: if a BGT polygon overlaps a BRT polygon
    # above this value we consider it "survived" generalization
    "survival_iou_threshold": 0.3,
}

print("CONFIG loaded")

## 1. Imports

In [ ]:
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

random.seed(42)
print("Imports OK")

## 2. Load tile index and sample tiles

In [ ]:
index_path = CONFIG["output_root"] / "tile_index.json"
with open(index_path) as f:
    tile_index = json.load(f)

print("Tile counts per split:")
for split, tiles in tile_index.items():
    print(f"  {split:5s}: {len(tiles)} tiles")

# Load a sample of tiles from each split for EDA
def load_sample_tiles(tile_index: dict, split: str, n: int,
                       bgt_col: str, brt_col: str) -> tuple:
    """
    Load n random tiles from a split.
    Returns (bgt_combined, brt_combined) GeoDataFrames.
    """
    tiles = tile_index[split]
    sample = random.sample(tiles, min(n, len(tiles)))

    bgt_frames, brt_frames = [], []
    for tile in sample:
        b = gpd.read_file(tile["bgt"])
        r = gpd.read_file(tile["brt"])
        b["tile"] = tile["bgt"]
        r["tile"] = tile["brt"]
        bgt_frames.append(b)
        brt_frames.append(r)

    bgt = gpd.GeoDataFrame(pd.concat(bgt_frames, ignore_index=True))
    brt = gpd.GeoDataFrame(pd.concat(brt_frames, ignore_index=True))
    return bgt, brt


n = CONFIG["eda_sample_tiles"]
bgt_train, brt_train = load_sample_tiles(tile_index, "train", n,
                                          CONFIG["bgt_class_col"],
                                          CONFIG["brt_class_col"])
print(f"\nLoaded {len(bgt_train):,} BGT polygons and {len(brt_train):,} BRT polygons from {n} train tiles")

## 3. Class distribution: BGT vs BRT

In [ ]:
bgt_col = CONFIG["bgt_class_col"]
brt_col = CONFIG["brt_class_col"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, gdf, col, title in [
    (axes[0], bgt_train, bgt_col, "BGT class distribution"),
    (axes[1], brt_train, brt_col, "BRT class distribution"),
]:
    if col in gdf.columns:
        counts = gdf[col].fillna("(missing)").value_counts().head(20)
        counts.plot.barh(ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Count")
        ax.invert_yaxis()
    else:
        ax.set_title(f"{title} — column '{col}' not found")

plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Also print raw counts
if bgt_col in bgt_train.columns:
    print("\nBGT top 10 classes:")
    print(bgt_train[bgt_col].value_counts().head(10).to_string())

## 4. Geometric properties

In [ ]:
def compute_geometric_features(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Compute per-polygon geometric statistics.
    Returns a DataFrame with area, perimeter, vertex count,
    compactness (circularity index), and bounding box aspect ratio.
    """
    df = pd.DataFrame()
    df["area"]          = gdf.geometry.area
    df["perimeter"]     = gdf.geometry.length

    # Vertex count (sum of exterior ring vertices)
    def vertex_count(geom):
        try:
            if geom.geom_type == "Polygon":
                return len(geom.exterior.coords)
            elif geom.geom_type == "MultiPolygon":
                return sum(len(p.exterior.coords) for p in geom.geoms)
        except Exception:
            return 0
        return 0

    df["n_vertices"]    = gdf.geometry.apply(vertex_count)

    # Compactness = 4π·area / perimeter²  (circle = 1.0, spiky shapes < 1)
    df["compactness"]   = (4 * np.pi * df["area"]) / (df["perimeter"] ** 2 + 1e-9)

    # Bounding box aspect ratio (width / height)
    bounds = gdf.geometry.bounds
    df["bbox_width"]    = bounds["maxx"] - bounds["minx"]
    df["bbox_height"]   = bounds["maxy"] - bounds["miny"]
    df["aspect_ratio"]  = df["bbox_width"] / (df["bbox_height"] + 1e-9)

    return df


bgt_geom = compute_geometric_features(bgt_train)
brt_geom = compute_geometric_features(brt_train)

print("BGT geometric stats:")
print(bgt_geom[["area", "perimeter", "n_vertices", "compactness"]].describe().round(2))
print("\nBRT geometric stats:")
print(brt_geom[["area", "perimeter", "n_vertices", "compactness"]].describe().round(2))

In [ ]:
# Compare distributions visually
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ("area",        "Area (m²)",         True),   # (column, label, log_scale)
    ("perimeter",   "Perimeter (m)",      True),
    ("n_vertices",  "Vertex count",       True),
    ("compactness", "Compactness index",  False),
]

for ax, (col, label, log) in zip(axes.flat, metrics):
    bgt_vals = bgt_geom[col].dropna()
    brt_vals = brt_geom[col].dropna()

    if log:
        bgt_vals = np.log1p(bgt_vals)
        brt_vals = np.log1p(brt_vals)
        label = f"log(1 + {label})"

    ax.hist(bgt_vals, bins=50, alpha=0.6, color="steelblue", label="BGT", density=True)
    ax.hist(brt_vals, bins=50, alpha=0.6, color="tomato",    label="BRT", density=True)
    ax.set_title(label)
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("BGT vs BRT — geometric feature distributions", fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "geometric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Polygon survival analysis

For each BGT polygon, check whether it has a corresponding BRT polygon
with sufficient IoU overlap. This is the *selection* task the model needs to learn.

In [ ]:
def compute_survival_rate(bgt: gpd.GeoDataFrame,
                           brt: gpd.GeoDataFrame,
                           iou_threshold: float) -> pd.DataFrame:
    """
    For each BGT polygon, find the best-matching BRT polygon by IoU.
    Labels each BGT polygon as 'survived' if max IoU >= iou_threshold.

    Returns BGT with added columns:
      - max_iou      : highest IoU with any BRT polygon
      - survived     : bool
    """
    # Spatial index join to find candidates
    joined = gpd.sjoin(bgt.reset_index(), brt[["geometry"]].reset_index(),
                       how="left", predicate="intersects",
                       lsuffix="bgt", rsuffix="brt")

    def iou(geom_a, geom_b):
        try:
            inter = geom_a.intersection(geom_b).area
            union = geom_a.union(geom_b).area
            return inter / union if union > 0 else 0.0
        except Exception:
            return 0.0

    # Compute IoU for each candidate pair
    brt_geoms = brt.geometry.values

    ious = []
    for _, row in joined.iterrows():
        brt_idx = row.get("index_brt")
        if pd.isna(brt_idx):
            ious.append(0.0)
        else:
            try:
                brt_geom = brt.geometry.iloc[int(brt_idx)]
                ious.append(iou(row.geometry, brt_geom))
            except Exception:
                ious.append(0.0)

    joined["iou"] = ious

    # Keep max IoU per BGT polygon
    max_iou = joined.groupby("index_bgt")["iou"].max()

    result = bgt.copy()
    result["max_iou"] = result.index.map(max_iou).fillna(0.0)
    result["survived"] = result["max_iou"] >= iou_threshold
    return result


print("Computing survival rates (this may take a minute for large samples)...")
bgt_with_survival = compute_survival_rate(
    bgt_train, brt_train, CONFIG["survival_iou_threshold"]
)

survived = bgt_with_survival["survived"].sum()
total    = len(bgt_with_survival)
print(f"\nSurvival rate: {survived:,} / {total:,} BGT polygons → {survived/total*100:.1f}%")

In [ ]:
# Survival rate per class
bgt_col = CONFIG["bgt_class_col"]

if bgt_col in bgt_with_survival.columns:
    survival_by_class = (
        bgt_with_survival.groupby(bgt_col)["survived"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "survived", "count": "total"})
    )
    survival_by_class["rate"] = survival_by_class["survived"] / survival_by_class["total"]
    survival_by_class = survival_by_class.sort_values("rate", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, len(survival_by_class) * 0.4)))
    survival_by_class["rate"].plot.barh(ax=ax, color="steelblue")
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="50% threshold")
    ax.set_title(f"BGT polygon survival rate by class (IoU ≥ {CONFIG['survival_iou_threshold']})")
    ax.set_xlabel("Fraction surviving into BRT")
    ax.legend()
    plt.tight_layout()
    plt.savefig(CONFIG["output_root"] / "survival_by_class.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nSurvival rate per class:")
    print(survival_by_class.to_string())

## 6. Visual comparison of a sample tile

In [ ]:
# Pick one tile from each split and show BGT vs BRT side by side
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col_idx, (split, tiles) in enumerate(tile_index.items()):
    if not tiles:
        continue
    tile = random.choice(tiles)
    bgt_t = gpd.read_file(tile["bgt"])
    brt_t = gpd.read_file(tile["brt"])

    # Top row: BGT
    ax_bgt = axes[0][col_idx]
    bgt_t.plot(ax=ax_bgt, color="steelblue", edgecolor="white", linewidth=0.3, alpha=0.8)
    ax_bgt.set_title(f"BGT — {split.upper()} ({tile['city']})")
    ax_bgt.set_axis_off()
    ax_bgt.annotate(f"{len(bgt_t)} polygons", xy=(0.02, 0.02),
                    xycoords="axes fraction", fontsize=9, color="white",
                    bbox=dict(boxstyle="round", fc="black", alpha=0.5))

    # Bottom row: BRT
    ax_brt = axes[1][col_idx]
    brt_t.plot(ax=ax_brt, color="tomato", edgecolor="white", linewidth=0.3, alpha=0.8)
    ax_brt.set_title(f"BRT — {split.upper()} ({tile['city']})")
    ax_brt.set_axis_off()
    ax_brt.annotate(f"{len(brt_t)} polygons", xy=(0.02, 0.02),
                    xycoords="axes fraction", fontsize=9, color="white",
                    bbox=dict(boxstyle="round", fc="black", alpha=0.5))

plt.suptitle("BGT (input) vs BRT (target) — random tile per split", fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "tile_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary statistics

In [ ]:
print("═" * 50)
print("EDA SUMMARY")
print("═" * 50)

for split, tiles in tile_index.items():
    print(f"\n{split.upper()} ({tiles[0]['city'] if tiles else 'none'})")
    print(f"  Tiles: {len(tiles)}")

print(f"\nBGT polygons in sample: {len(bgt_train):,}")
print(f"BRT polygons in sample: {len(brt_train):,}")
print(f"Reduction ratio (BRT/BGT): {len(brt_train)/len(bgt_train):.2f}")
print(f"\nBGT median area  : {bgt_geom['area'].median():.1f} m²")
print(f"BRT median area  : {brt_geom['area'].median():.1f} m²")
print(f"BGT median verts : {bgt_geom['n_vertices'].median():.0f}")
print(f"BRT median verts : {brt_geom['n_vertices'].median():.0f}")

if "survived" in bgt_with_survival.columns:
    print(f"\nPolygon survival rate: {bgt_with_survival['survived'].mean()*100:.1f}%")

print("\nAll EDA plots saved to:", CONFIG["output_root"])